In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from typing import Literal


import pandas


df = pandas.read_json("data/gooaq/gooaq.jsonl", lines=True, chunksize=10_000)
for chunk in df:
    print(chunk.head())
    break

   id                             question      short_answer answer  \
0   1          0800 da maquina do sicredi?               NaN    NaN   
1   2            0800 da natura do brasil?  1 (720) 408-2293    NaN   
2   3  0800 da ouvidoria do banco central?               NaN    NaN   
3   4   0800 da receita federal do brasil?               NaN    NaN   
4   5           0800 da renault do brasil?               NaN    NaN   

  answer_type answer_url  
0     unknown        NaN  
1   knowledge        NaN  
2     unknown        NaN  
3     unknown        NaN  
4     unknown        NaN  


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="data/gooaq/gooaq.jsonl",   # or "large.json"
    split="train",
)

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset

Dataset({
    features: ['id', 'question', 'short_answer', 'answer', 'answer_type', 'answer_url'],
    num_rows: 5030530
})

In [5]:
idx = 0 

for data_row in dataset:
    print(data_row)
    idx += 1
    if idx > 100:
        break

{'id': 1, 'question': '0800 da maquina do sicredi?', 'short_answer': None, 'answer': None, 'answer_type': 'unknown', 'answer_url': None}
{'id': 2, 'question': '0800 da natura do brasil?', 'short_answer': '1 (720) 408-2293', 'answer': None, 'answer_type': 'knowledge', 'answer_url': None}
{'id': 3, 'question': '0800 da ouvidoria do banco central?', 'short_answer': None, 'answer': None, 'answer_type': 'unknown', 'answer_url': None}
{'id': 4, 'question': '0800 da receita federal do brasil?', 'short_answer': None, 'answer': None, 'answer_type': 'unknown', 'answer_url': None}
{'id': 5, 'question': '0800 da renault do brasil?', 'short_answer': None, 'answer': None, 'answer_type': 'unknown', 'answer_url': None}
{'id': 6, 'question': '0800 da toledo do brasil?', 'short_answer': None, 'answer': None, 'answer_type': 'unknown', 'answer_url': None}
{'id': 7, 'question': '0800 da uber do brasil?', 'short_answer': None, 'answer': None, 'answer_type': 'unknown', 'answer_url': None}
{'id': 8, 'question

In [6]:
from qdrant_client import QdrantClient
import os

client = QdrantClient(
    url=os.getenv('QDRANT_CLOUD_URL'),
    api_key=os.getenv('QDRANT_CLOUD_API_KEY'),
    cloud_inference=True
)

In [7]:
client.cloud_inference

True

In [8]:
from hybrid_search_rrf_dataset.indexer import EmbeddingConfig
from qdrant_client.models import Distance

CLOUD_DENSE_MODEL = 'mixedbread-ai/mxbai-embed-large-v1'
SIZE = 1024


dense_config = EmbeddingConfig(
    name='dense_base',
    model_id=CLOUD_DENSE_MODEL,
    kind='dense',
    size=SIZE,
    distance=Distance.COSINE,
)

embeddings_config = dense_config

In [9]:
from hybrid_search_rrf_dataset.indexer import CorpusIndexer

indexer = CorpusIndexer(client, 'gooaq', [embeddings_config])

In [10]:
indexer.ensure_collection()

In [ ]:
def gooaq_to_doc(row: dict) -> dict:
    return {
        "doc_id": str(row["id"]),
        "title": "",
        "text": row["question"],
        "metadata": {
            "short_answer": row["short_answer"],
            "answer": row["answer"],
            "answer_type": row["answer_type"],
            "answer_url": row["answer_url"],
        },
    }


indexer.upload_points_iter(
    (gooaq_to_doc(row) for row in dataset),
    batch_size=256,
    parallel=4,
)

upload:gooaq:   2%|▏         | 98560/5030530 [10:00<8:02:14, 170.45it/s] 

In [ ]:
indexer.upload_points_iter(
    (gooaq_to_doc(row) for row in dataset),
    batch_size=256,
    parallel=4,
)